# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [ ]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [15]:
# === CONFIG: point this to the GenAI Divide PDF on your machine ===
pdf_path = "data/The-GenAI-Divide-State-of-AI-in-Business-2025.pdf"  # <-- change as needed

# A title used downstream (generation/evaluation cells)
document_title = "The GenAI Divide: State of AI in Business 2025"

In [16]:
import os, sys, glob, pathlib

print("CWD:", os.getcwd())
print("Python:", sys.version)
print("\nNearby PDFs (depth=2):")
for p in glob.glob("**/*.pdf", recursive=True):
    if p.count(os.sep) <= 2:  # keep list readable
        print(" -", p)

CWD: c:\Users\user\Desktop\DSI\deploying-ai\02_activities
Python: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]

Nearby PDFs (depth=2):
 - documents\ai_report_2025.pdf
 - documents\managing_oneself.pdf


In [17]:
# EITHER: point to your local file (use a raw string on Windows)
# Example if it sits next to your notebook:
# pdf_path = "The-GenAI-Divide-State-of-AI-in-Business-2025.pdf"

# Example absolute Windows path:
# pdf_path = r"C:\Users\user\Desktop\DSI\deploying-ai\data\The-GenAI-Divide-State-of-AI-in-Business-2025.pdf"

# Example project-relative path:
# pdf_path = "data/The-GenAI-Divide-State-of-AI-in-Business-2025.pdf"

# OR: paste a direct PDF URL:
# pdf_path = "https://example.com/The-GenAI-Divide-State-of-AI-in-Business-2025.pdf"

pdf_path = "data/The-GenAI-Divide-State-of-AI-in-Business-2025.pdf"  # <-- change me

document_title = "The GenAI Divide: State of AI in Business 2025"

In [18]:
# === Auto-locate "The GenAI Divide: State of AI in Business 2025" PDF ===
import os, sys, glob, pathlib, re

NOTEBOOK_DIR = pathlib.Path.cwd()
SEARCH_ROOTS = [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents)[:4]   # search here and up to 4 parents

# Common name fragments / variants to match
NAME_HINTS = [
    "genai divide", "genai-divide", "genai_divide",
    "state of ai in business 2025", "state-of-ai-in-business-2025",
    "genai", "state of ai", "ai in business 2025"
]

def score_path(p: pathlib.Path) -> int:
    s = p.name.lower()
    score = 0
    for h in NAME_HINTS:
        if h in s:
            score += 5
    # Prefer shorter names and direct "genai divide" matches
    if "genai" in s and "divide" in s:
        score += 10
    score += max(0, 60 - len(s)) // 10
    return score

candidates = []
for root in SEARCH_ROOTS:
    for p in root.rglob("*.pdf"):
        candidates.append(p)

# Sort by heuristic score, then by depth (shallower first)
candidates = sorted(candidates, key=lambda p: (-score_path(p), len(p.parts)))

if not candidates:
    raise FileNotFoundError(
        "No PDFs found near the notebook. Options:\n"
        "  1) Put the PDF in this folder and set pdf_path = '<filename>.pdf'\n"
        "  2) Create a 'data' folder next to your notebook and put the PDF there.\n"
        "  3) Paste an absolute path (e.g., r'C:\\Users\\user\\Desktop\\...pdf')."
    )

# Filter to likely matches if possible
likely = [p for p in candidates if score_path(p) >= 10] or candidates[:10]

print("Found PDF candidates (top 10, best match first):")
for i, p in enumerate(likely[:10]):
    print(f"[{i}] {p}")

# >>> If the first one looks correct, leave CHOICE = 0. Otherwise, change CHOICE to the right index and re-run this cell.
CHOICE = 0

pdf_path = str(likely[CHOICE])
document_title = "The GenAI Divide: State of AI in Business 2025"
print("\nUsing pdf_path =", pdf_path)


Found PDF candidates (top 10, best match first):
[0] c:\Users\user\Documents\800.pdf
[1] c:\Users\user\Documents\D1.pdf
[2] c:\Users\user\Documents\D2.pdf
[3] c:\Users\user\Documents\Doc4.pdf
[4] c:\Users\user\Downloads\1.pdf
[5] c:\Users\user\Downloads\10 a b.pdf
[6] c:\Users\user\Downloads\7.pdf
[7] c:\Users\user\Downloads\800.pdf
[8] c:\Users\user\Downloads\ch4.pdf
[9] c:\Users\user\Downloads\ch5.pdf

Using pdf_path = c:\Users\user\Documents\800.pdf


In [19]:
# === Robust PDF loader: LangChain PyPDFLoader → pypdf fallback ===
import os, re

def _clean_page_text(t: str) -> str:
    t = t.replace("\u00ad", "")                  # soft hyphen
    t = re.sub(r"-\n(?=\w)", "", t)             # join hyphenated line breaks
    t = re.sub(r"\n{2,}", "\n\n", t)            # collapse extra blank lines
    t = re.sub(r"[ \t]+", " ", t)               # collapse repeated spaces
    return t.strip()

if not os.path.isfile(pdf_path):
    raise FileNotFoundError(
        f"PDF not found at:\n  {pdf_path}\n"
        f"Current working dir:\n  {os.getcwd()}\n"
        "Tip: Adjust CHOICE in the previous cell, or set pdf_path to an absolute path."
    )

document_text = ""
source_used = None
docs = None

try:
    from langchain_community.document_loaders import PyPDFLoader
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    document_text = ""
    for page in docs:
        document_text += _clean_page_text(page.page_content) + "\n"
    source_used = "langchain_community.document_loaders.PyPDFLoader"
except Exception as e_lc:
    try:
        from pypdf import PdfReader
        reader = PdfReader(pdf_path)
        pages = [p.extract_text() or "" for p in reader.pages]
        document_text = ""
        for ptxt in pages:
            document_text += _clean_page_text(ptxt) + "\n"
        source_used = "pypdf.PdfReader"
    except Exception as e_pp:
        raise RuntimeError(
            "Failed to load PDF via LangChain and pypdf.\n"
            f"LangChain error: {e_lc}\n"
            f"pypdf error: {e_pp}\n"
            "Fix the path or install the missing library."
        )

print(f"Loaded PDF with: {source_used}")
print(f"Characters: {len(document_text):,}")
print("\nPreview:\n", document_text[:800] + ("..." if len(document_text) > 800 else ""))

# (Optional) If your template expects the explicit 'docs join' loop:
if docs:
    tmp_text = ""
    for page in docs:
        tmp_text += page.page_content + "\n"
    # Keep the cleaned version as the canonical text
    document_text = _clean_page_text(tmp_text)

Loaded PDF with: langchain_community.document_loaders.PyPDFLoader
Characters: 7

Preview:
 









In [20]:
pdf_path = r"C:\Users\user\Desktop\DSI\deploying-ai\02_activities\The-GenAI-Divide-State-of-AI-in-Business-2025.pdf"

In [21]:
import os, re, io, pathlib
from typing import Optional, Tuple

def _clean_text(t: str) -> str:
    if not t:
        return ""
    t = t.replace("\u00ad", "")             # soft hyphen
    t = re.sub(r"-\n(?=\w)", "", t)         # join hyphenated line breaks
    t = re.sub(r"[ \t]+", " ", t)           # collapse spaces
    t = re.sub(r"\n{3,}", "\n\n", t)        # collapse extra blank blocks
    return t.strip()

def try_langchain_pypdfloader(pdf_path: str) -> Tuple[str, str]:
    from langchain_community.document_loaders import PyPDFLoader
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    text = "\n".join(page.page_content or "" for page in docs)
    return _clean_text(text), "langchain.PyPDFLoader"

def try_pypdf(pdf_path: str) -> Tuple[str, str]:
    from pypdf import PdfReader
    reader = PdfReader(pdf_path)
    parts = []
    for p in reader.pages:
        parts.append(p.extract_text() or "")
    return _clean_text("\n".join(parts)), "pypdf.PdfReader"

def try_pdfminer(pdf_path: str) -> Tuple[str, str]:
    from pdfminer.high_level import extract_text
    text = extract_text(pdf_path) or ""
    return _clean_text(text), "pdfminer.six"

def try_pymupdf(pdf_path: str) -> Tuple[str, str]:
    import fitz  # pymupdf
    doc = fitz.open(pdf_path)
    parts = []
    for page in doc:
        # "text" is raw layout text; "blocks" could work too: page.get_text("blocks")
        parts.append(page.get_text("text") or "")
    return _clean_text("\n".join(parts)), "pymupdf.get_text('text')"

def ocr_with_pypdfium2_tesseract(pdf_path: str, dpi: int = 220) -> Tuple[str, str]:
    import pypdfium2 as pdfium
    from PIL import Image
    import pytesseract

    pdf = pdfium.PdfDocument(pdf_path)
    text_parts = []
    for i in range(len(pdf)):
        page = pdf.get_page(i)
        # Render to bitmap
        pil_image = page.render(scale=dpi/72).to_pil()
        # OCR
        txt = pytesseract.image_to_string(pil_image)
        text_parts.append(txt)
    return _clean_text("\n".join(text_parts)), f"pypdfium2 render + Tesseract OCR @ {dpi} dpi"

def extract_pdf_text_robust(pdf_path: str) -> Tuple[str, str]:
    errors = []
    # 1) langchain
    try:
        t, src = try_langchain_pypdfloader(pdf_path)
        if len(t) > 400:  # plenty of text
            return t, src
    except Exception as e:
        errors.append(("langchain", str(e)))
    # 2) pypdf
    try:
        t, src = try_pypdf(pdf_path)
        if len(t) > 400:
            return t, src
    except Exception as e:
        errors.append(("pypdf", str(e)))
    # 3) pdfminer
    try:
        t, src = try_pdfminer(pdf_path)
        if len(t) > 400:
            return t, src
    except Exception as e:
        errors.append(("pdfminer", str(e)))
    # 4) pymupdf
    try:
        t, src = try_pymupdf(pdf_path)
        if len(t) > 400:
            return t, src
    except Exception as e:
        errors.append(("pymupdf", str(e)))
    # 5) last resort: OCR
    try:
        t, src = ocr_with_pypdfium2_tesseract(pdf_path, dpi=220)
        if len(t) > 100:  # OCR text can be shorter/fragmented; lower threshold
            return t, src
    except Exception as e:
        errors.append(("ocr", str(e)))

    # If all failed or produced only tiny text, still return best attempt (maybe blank)
    return t if 't' in locals() else "", f"failed_all ({errors})"

# --- run it ---
assert 'pdf_path' in globals(), "Please set `pdf_path` first (see earlier cell)."
document_text, source_used = extract_pdf_text_robust(pdf_path)

print("Extractor used:", source_used)
print("Characters:", len(document_text))
print("\nPreview:\n", document_text[:1000] + ("..." if len(document_text) > 1000 else ""))

# Keep your existing title variable
document_title = "The GenAI Divide: State of AI in Business 2025"

Extractor used: failed_all ([('langchain', 'File path C:\\Users\\user\\Desktop\\DSI\\deploying-ai\\02_activities\\The-GenAI-Divide-State-of-AI-in-Business-2025.pdf is not a valid file or url'), ('pypdf', "[Errno 2] No such file or directory: 'C:\\\\Users\\\\user\\\\Desktop\\\\DSI\\\\deploying-ai\\\\02_activities\\\\The-GenAI-Divide-State-of-AI-in-Business-2025.pdf'"), ('pdfminer', "No module named 'pdfminer'"), ('pymupdf', "No module named 'fitz'"), ('ocr', "No module named 'pypdfium2'")])
Characters: 0

Preview:
 


In [22]:
import re

def strip_boilerplate(text: str) -> str:
    t = text
    # Remove common headings for TOC or references if they dominate
    t = re.sub(r"\n\s*table of contents\s*\n.*?\n\n", "\n\n", t, flags=re.I|re.S)
    t = re.sub(r"\n\s*references\s*\n.*?$", "\n", t, flags=re.I|re.S)
    # Remove repeated headers/footers (simple heuristic: short lines that repeat many times)
    lines = t.splitlines()
    counts = {}
    for ln in lines:
        ln2 = ln.strip().lower()
        if 0 < len(ln2) <= 60:
            counts[ln2] = counts.get(ln2, 0) + 1
    frequent = {k for k,v in counts.items() if v >= 5}
    cleaned = []
    for ln in lines:
        if ln.strip().lower() in frequent:
            continue
        cleaned.append(ln)
    t = "\n".join(cleaned)
    # Collapse excessive blank lines again
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

if len(document_text) < 500:
    print("Warning: extracted text is very short; proceeding without boilerplate stripping.")
else:
    document_text = strip_boilerplate(document_text)
    print("After boilerplate stripping, characters:", len(document_text))

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [24]:
 #=== Generation Task: Setup ===
from __future__ import annotations
from typing import Optional
from pydantic import BaseModel, Field, ValidationError
import json
import os

# ---- Choose a model that is NOT in the GPT-5 family ----
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")  # e.g., "gpt-4o-mini", "gpt-4o", "o4-mini"
assert not OPENAI_MODEL.startswith("gpt-5"), "Use a non-GPT-5 model as required."

# ---- Distinct, identifiable summary tone ----
SUMMARY_TONE = "Formal Academic Writing"   # e.g., "Victorian English", "AAVE", "Bureaucratese", etc.

# ---- Pydantic structured output ----
class SummaryOutput(BaseModel):
    Author: str = Field(..., description="Author(s) of the article, if identifiable from the text/context.")
    Title: str = Field(..., description="Title of the article.")
    Relevance: str = Field(..., description="Why this article matters to AI professionals (<= 1 paragraph).")
    Summary: str = Field(..., description="Concise summary (<= 1000 tokens).")
    Tone: str = Field(..., description="The style/tone used to write the summary.")
    InputTokens: int = Field(..., description="Input token count from the response object.")
    OutputTokens: int = Field(..., description="Output token count from the response object.")

# ---- Developer (instructions) prompt, stored separately ----
DEV_INSTRUCTIONS = """\
You are a helpful assistant that produces structured JSON only.
Follow the required fields exactly and write the summary in the specified tone.
Keep the Relevance to a single paragraph. The Summary must be concise (<= 1000 tokens).
If author(s) is/are not obvious, infer reasonably or say "Unknown".
Return ONLY a JSON object with the specified keys, no extra text.
"""

# ---- User prompt template (context is injected dynamically) ----
USER_TASK_TEMPLATE = """\
Task: Produce a structured summary for the given article.

Tone to use for the “Summary”: {tone}

Required JSON keys (exactly):
- Author
- Title
- Relevance (<= 1 paragraph)
- Summary (<= 1000 tokens)
- Tone
- InputTokens
- OutputTokens

Article Title (if available): {doc_title}

Article Content:
\"\"\"{doc_text}\"\"\"
"""

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [34]:
def overlap_score(a_words: List[str], b_words: List[str]) -> float:
    from collections import Counter
    a, b = Counter(a_words), Counter(b_words)
    inter = sum((a & b).values())
    p = inter / max(1, sum(b.values()))
    r = inter / max(1, sum(a.values()))
    return 0.0 if (p + r) == 0 else 2 * p * r / (p + r)

def readability_score(text: str) -> float:
    sents = sent_tokenize(text)
    if not sents: return 0.0
    lens = [len(word_tokenize(s)) for s in sents]
    avg = sum(lens) / len(lens)
    return max(0.0, min(1.0, 25.0 / max(25.0, avg)))  # <=25 words/sent ~ good

def tonality_match(summary: str, tone: str) -> float:
    academic_markers = ["in summary", "moreover", "furthermore", "consequently", "thus", "therefore"]
    slang = ["lol", "gonna", "wanna", "ain't"]
    score = 0.5 + 0.1 * sum(1 for m in academic_markers if m in summary.lower())
    score -= 0.2 * sum(1 for s in slang if s in summary.lower())
    return max(0.0, min(1.0, score))

def safety_check(text: str) -> float:
    bad = ["hate speech", "kill", "slur", "violent", "nsfw", "racist", "sexist"]
    return 1.0 if not any(b in text.lower() for b in bad) else 0.2

In [ ]:
# Five questions each
SUMM_QUESTIONS = [
    "Does the summary cover the thesis about noise as context-dependent perception?",
    "Does it relate noise to decision-making/cognitive interference?",
    "Does it acknowledge the value of variance and risk of suppressing novelty?",
    "Does it avoid hallucinating details not in context?",
    "Is it concise and within budget?",
]
COHERENCE_QUESTIONS = [
    "Are sentences logically connected?",
    "Is the flow from definition to implications clear?",
    "Are pronouns/references unambiguous?",
    "Does the structure avoid contradictions?",
    "Is it easy to follow on first read?",
]
TONALITY_QUESTIONS = [
    "Does it read as Formal Academic Writing?",
    "Are discourse markers appropriate?",
    "Is slang avoided?",
    "Is hedging/modality appropriate?",
    "Is tone consistent across sentences?",
]
SAFETY_QUESTIONS = [
    "No hateful/harassing content?",
    "No unsafe instructions?",
    "Sensitive content handled neutrally?",
    "Professional/respectful language?",
    "No explicit/graphic material?",
]

def score_summarization(context: str, summary: str) -> Tuple[float, str]:
    c_words = word_tokenize(context); s_words = word_tokenize(summary)
    f1 = overlap_score(c_words, s_words)
    concision = min(1.0, 180 / max(1, len(s_words)))
    rare = [w for w in set(s_words) if w not in set(c_words)]
    halluc_proxy = max(0.0, 1.0 - 0.01 * len(rare))
    score = 0.5*f1 + 0.3*concision + 0.2*halluc_proxy
    reason = f"F1(overlap)={f1:.2f}, concision={concision:.2f}, hallucination_proxy={halluc_proxy:.2f}, rare_terms={len(rare)}."
    return score, reason

def score_coherence(summary: str) -> Tuple[float, str]:
    r = readability_score(summary)
    connectives = sum(summary.lower().count(k) for k in ["in summary", "moreover", "furthermore", "consequently"])
    score = max(0.0, min(1.0, 0.6*r + 0.1*connectives + 0.3))
    reason = f"Readability={r:.2f}, connectives={connectives}."
    return score, reason

def score_tonality(summary: str, tone: str) -> Tuple[float, str]:
    t = tonality_match(summary, tone)
    reason = f"Tone match for '{tone}' = {t:.2f}."
    return t, reason

def score_safety(summary: str) -> Tuple[float, str]:
    s = safety_check(summary)
    reason = "No flagged unsafe terms detected." if s >= 1.0 else "Potentially unsafe terms present."
    return s, reason

def aggregate_questions(score: float, questions: List[str]) -> Dict[str, List[str]]:
    return {"questions": questions, "overall_score": round(score, 3)}

summ_score, summ_reason = score_summarization(DOCUMENT_TEXT, summary_text)
coh_score,  coh_reason  = score_coherence(summary_text)
tone_score, tone_reason = score_tonality(summary_text, SUMMARY_TONE)
safe_score, safe_reason = score_safety(summary_text)

EVAL_RESULTS = {
    "SummarizationScore": round(summ_score, 3),
    "SummarizationReason": summ_reason,
    "CoherenceScore": round(coh_score, 3),
    "CoherenceReason": coh_reason,
    "TonalityScore": round(tone_score, 3),
    "TonalityReason": tone_reason,
    "SafetyScore": round(safe_score, 3),
    "SafetyReason": safe_reason,
    "SummarizationAssessment": aggregate_questions(summ_score, SUMM_QUESTIONS),
    "CoherenceAssessment": aggregate_questions(coh_score, COHERENCE_QUESTIONS),
    "TonalityAssessment": aggregate_questions(tone_score, TONALITY_QUESTIONS),
    "SafetyAssessment": aggregate_questions(safe_score, SAFETY_QUESTIONS),
}

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
def enhance_summary(context: str, summary: str, tone: str, eval_results: Dict) -> str:
    added = []
    if eval_results["SummarizationScore"] < 0.7:
        added.append("This analysis emphasizes that noise is a culturally framed perception and that variance may encode useful information.")
        added.append("Accordingly, the practical aim is to design evaluative systems that minimize cognitive interference without suppressing novelty.")
    if eval_results["TonalityScore"] < 0.8 and tone.lower().startswith("formal academic"):
        added.append("Therefore, the discussion adopts a formal register and employs standard academic connectives to maintain clarity.")
    candidate = summary
    if added:
        candidate = candidate.rstrip()
        if not candidate.endswith("."): candidate += "."
        candidate += " " + " ".join(added)
    # enforce token budget
    words = word_tokenize(candidate)
    if len(words) > 180:
        candidate = " ".join(words[:180])
    return candidate

enhanced_summary = enhance_summary(DOCUMENT_TEXT, summary_text, SUMMARY_TONE, EVAL_RESULTS)

# Re-evaluate enhanced
def evaluate_all(context: str, summary: str, tone: str) -> Dict[str, float | str]:
    s1, r1 = score_summarization(context, summary)
    s2, r2 = score_coherence(summary)
    s3, r3 = score_tonality(summary, tone)
    s4, r4 = score_safety(summary)
    return {
        "SummarizationScore": round(s1, 3), "SummarizationReason": r1,
        "CoherenceScore": round(s2, 3), "CoherenceReason": r2,
        "TonalityScore": round(s3, 3), "TonalityReason": r3,
        "SafetyScore": round(s4, 3), "SafetyReason": r4
    }

ENH_EVAL_RESULTS = evaluate_all(DOCUMENT_TEXT, enhanced_summary, SUMMARY_TONE)

In [ ]:
out_dir = Path("./deploying_ai_assignment_1_artifacts")
out_dir.mkdir(parents=True, exist_ok=True)

def as_json_obj(obj):
    if hasattr(obj, "model_dump"): return obj.model_dump()
    try: return asdict(obj)
    except: return obj.__dict__

with open(out_dir/"structured_summary.json", "w", encoding="utf-8") as f:
    json.dump(as_json_obj(structured), f, ensure_ascii=False, indent=2)

with open(out_dir/"evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(EVAL_RESULTS, f, ensure_ascii=False, indent=2)

with open(out_dir/"enhanced_evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(ENH_EVAL_RESULTS, f, ensure_ascii=False, indent=2)

report = f"""# Deploying AI — Assignment 1: Evaluating Summaries

## Document
- **Title:** {DOCUMENT_TITLE}
- **Author:** {DOCUMENT_AUTHOR}

## Prompts
**Developer Instructions**

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
